# 03_build_indicators.py 결과 확인

`data/indicators`에 저장된 지표 계산 결과를 확인한다.
- 전체 건수, 컬럼별 결측 비율
- 종목별 지표 스냅샷 (EPS/BPS/PER/PBR/ROE/부채비율/배당수익률/ROA/모멘텀/F-Score/EPS성장률)
- 이상치(완전자본잠식 등 N/A 처리된 종목) 확인

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_indicators").getOrCreate()

df = spark.read.parquet("/opt/spark-apps/data/indicators")
total = df.count()
print(f"전체 {total}건")
df.printSchema()

전체 86건
root
 |-- stock_code: string (nullable = true)
 |-- fs_div: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- eps: double (nullable = true)
 |-- bps: double (nullable = true)
 |-- per: double (nullable = true)
 |-- pbr: double (nullable = true)
 |-- roe: double (nullable = true)
 |-- debt_ratio: double (nullable = true)
 |-- dividend_yield: double (nullable = true)
 |-- roa: double (nullable = true)
 |-- momentum: double (nullable = true)
 |-- f_score: integer (nullable = true)
 |-- eps_growth_rate: double (nullable = true)
 |-- year: integer (nullable = true)



## 1. 종목별 지표 스냅샷 (표로 보기)

In [2]:
cols = ["stock_code", "eps", "bps", "per", "pbr", "roe", "debt_ratio",
        "dividend_yield", "roa", "momentum", "f_score", "eps_growth_rate"]

pdf = df.select(*cols).orderBy("stock_code").toPandas()
pdf

,stock_code,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,000020,1010.94,14573.40,4.73,0.33,6.94,38.79,3.77,5.00,-26.81,6,30.78
1,000040,-1217.17,1296.94,NaN,0.81,-93.85,518.49,NaN,-15.17,-46.72,2,NaN
2,000050,-490.68,26920.97,NaN,0.30,-1.82,63.52,NaN,-1.11,6.91,5,-404.67
3,000070,29242.36,360230.26,1.93,0.16,8.12,83.68,6.19,4.42,-36.24,6,140.83
4,000080,506.34,15910.71,30.08,0.96,3.18,200.96,6.24,1.06,-26.44,6,-59.10
...,...,...,...,...,...,...,...,...,...,...,...,...
81,900270,-38.16,14881.70,NaN,0.03,-0.26,16.71,NaN,-0.22,169.53,4,NaN
82,900290,819.43,11263.29,3.61,0.26,7.28,92.80,0.00,3.77,-7.11,3,30.72
83,900300,-12075.99,39782.55,NaN,0.04,-30.36,31.70,0.00,-23.05,356.69,3,NaN
84,900310,488.22,23215.58,3.51,0.07,2.10,17.51,NaN,1.79,140.16,4,138.37


## 2. 컬럼별 결측 비율 (%)
PER/배당수익률/EPS성장률은 적자기업·전년도 재무제표 없음 등으로 원래도 일정 비율 NULL이 나오는 게 정상.

In [3]:
metric_cols = ["eps", "bps", "per", "pbr", "roe", "debt_ratio",
               "dividend_yield", "roa", "momentum", "f_score", "eps_growth_rate"]

null_ratio = df.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 1).alias(c)
    for c in metric_cols
])
null_ratio.toPandas()

,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,0.0,1.2,31.4,1.2,1.2,1.2,27.9,0.0,0.0,0.0,27.9


## 3. 완전자본잠식 종목 (equity<=0 이라 BPS/PBR/ROE/부채비율이 전부 N/A인 케이스)
EPS는 있는데 BPS가 NULL이면 자본총계가 0 이하라는 뜻 (FinancialIndicatorService.calculate와 동일 조건).

In [4]:
capital_impaired = df.filter(F.col("eps").isNotNull() & F.col("bps").isNull())
print(f"완전자본잠식 추정 종목: {capital_impaired.count()}건")
capital_impaired.select(*cols).toPandas()

완전자본잠식 추정 종목: 1건


,stock_code,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,000300,-4275.48,NaN,NaN,NaN,NaN,NaN,NaN,-238.53,111.69,4,NaN


## 4. 지표 분포 요약 (min/max/mean 등) - 극단치 스캔용

In [5]:
df.select(*metric_cols).summary("min", "25%", "50%", "75%", "max").toPandas()

,summary,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,min,-12820.99,115.52,0.62,0.03,-256.6,6.62,0.0,-238.53,-92.64,2,-516.37
1,25%,-128.69,8291.64,3.98,0.21,-1.01,34.25,0.0,-0.79,-21.55,4,-74.58
2,50%,488.22,17140.32,7.9,0.41,3.25,90.51,1.73,1.38,-2.89,5,-24.36
3,75%,1972.23,56858.04,14.93,0.91,6.38,185.34,4.21,3.45,26.42,7,48.55
4,max,40799.94,692037.34,252.55,20.65,24.01,1081.55,30.03,12.93,1230.12,9,5981.62


In [6]:
spark.stop()